# RingID / Tree-Ring Benchmark — Google Colab

**Generation-based** (SD2.1 + spectral latent ring). Runs **14 WAVES attacks** from `benchmark_attacks.py`.

**Metrics:** PSNR, SSIM, AUROC, TPR@1% FPR (detector = DDIM invert → ℓ1 distance to key; score = −distance).

### Setup
1. Runtime → **GPU** (T4 works; use smaller batch sizes if OOM)
2. Run all cells — clones [wavess](https://github.com/ademladhari/wavess) + [tree-ring-ringid](https://github.com/ademladhari/tree-ring-ringid)
3. Optional: Hugging Face login if SD2.1 download is gated
4. Optional: mount Drive in cell 1 to save results

In [ ]:
# ── 1. Clone repos + optional Drive ─────────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

WAVES_URL    = 'https://github.com/ademladhari/wavess.git'
RINGID_URL   = 'https://github.com/ademladhari/tree-ring-ringid.git'
WAVES_ROOT   = Path('/content/wavess')
RINGID_ROOT  = WAVES_ROOT / 'tree-ring-ringid'

if not WAVES_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', WAVES_URL, str(WAVES_ROOT)], check=True)
else:
    print('wavess already present')

if not RINGID_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', RINGID_URL, str(RINGID_ROOT)], check=True)
else:
    print('tree-ring-ringid already present')

for p in (WAVES_ROOT, RINGID_ROOT, RINGID_ROOT / 'src'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

_ba = WAVES_ROOT / 'benchmark_attacks.py'
_bench = WAVES_ROOT / 'ringid_benchmark.py'
if not _ba.is_file():
    raise FileNotFoundError(f'Missing {_ba} — push benchmark_attacks.py to wavess on GitHub')
if not _bench.is_file():
    raise FileNotFoundError(
        f'Missing {_bench}\n'
        'Pull latest wavess (contains ringid_benchmark.py), then re-run:\n'
        '  !rm -rf /content/wavess   # then re-run this cell'
    )

os.chdir(WAVES_ROOT)

SAVE_TO_DRIVE = True
DRIVE_OUTPUT  = '/content/drive/MyDrive/wavess_results/ringid'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print('OK:', RINGID_ROOT)

In [ ]:
# ── 2. Install dependencies ─────────────────────────────────────────────────
!pip install -q -U pip
!pip install -q -r {RINGID_ROOT}/requirements.txt scikit-image scikit-learn tqdm pandas
!pip install -q -e {RINGID_ROOT}

import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 3. Hugging Face login (only if SD2.1 download fails) ───────────────────
HF_LOGIN = False  # set True if you hit gated-model errors

if HF_LOGIN:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print('Skipping HF login (set HF_LOGIN=True if needed)')

In [ ]:
# ── 4. Config — edit here ───────────────────────────────────────────────────
N_IMAGES         = 100     # watermarked + clean pairs
PROFILE          = 'ring_id'   # 'ring_id' (RingID) or 'tree_ring' (Tree-Ring baseline)
MODEL_ID         = 'sd2-community/stable-diffusion-2-1-base'
DTYPE            = 'float16'   # float32 if unstable on T4
NUM_STEPS        = 50
INVERSION_STEPS  = None        # None → same as NUM_STEPS
GEN_BATCH_SIZE   = 2           # lower to 1 if OOM during generation
DETECT_BATCH_SIZE = 1          # lower to 1 if OOM during DDIM invert
SEED             = 42
WATERMARK_SEED   = 42
HEIGHT           = 512
WIDTH            = 512

PROMPTS_FILE     = str(WAVES_ROOT / 'tree-ring' / 'prompts.txt')
OUTPUT_DIR       = str(RINGID_ROOT / 'outputs_benchmark_colab')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'Profile={PROFILE}  n={N_IMAGES}  gen_batch={GEN_BATCH_SIZE}  detect_batch={DETECT_BATCH_SIZE}')

In [ ]:
# ── 5. Run benchmark (14 WAVES attacks) ─────────────────────────────────────
# Uses wavess/ringid_benchmark.py (mainbenchmark is NOT on the ringid GitHub clone)
import subprocess

BENCH_SCRIPT = str(WAVES_ROOT / 'ringid_benchmark.py')
cmd = [
    sys.executable, BENCH_SCRIPT,
    '--output', OUTPUT_DIR,
    '--n-images', str(N_IMAGES),
    '--prompts', PROMPTS_FILE,
    '--seed', str(SEED),
    '--watermark-seed', str(WATERMARK_SEED),
    '--profile', PROFILE,
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--steps', str(NUM_STEPS),
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--gen-batch-size', str(GEN_BATCH_SIZE),
    '--detect-batch-size', str(DETECT_BATCH_SIZE),
]
if INVERSION_STEPS is not None:
    cmd.extend(['--inversion-steps', str(INVERSION_STEPS)])

print(' '.join(cmd))
rc = subprocess.run(cmd, cwd=str(WAVES_ROOT)).returncode
if rc != 0:
    raise RuntimeError(f'ringid_benchmark.py failed (exit {rc}). See traceback above.')

In [ ]:
# ── 6. Results table + optional Drive sync ────────────────────────────────────
import json, shutil
import pandas as pd
from pathlib import Path

out = Path(OUTPUT_DIR)
csv_path = out / 'results.csv'
if not csv_path.is_file():
    raise FileNotFoundError(f'No results at {csv_path} — did cell 5 finish?')

df = pd.read_csv(csv_path)
show = df[['attack', 'PSNR', 'SSIM', 'AUROC', 'TPR_at_1pct_FPR']].copy()
show.columns = ['Attack', 'PSNR', 'SSIM', 'AUROC', 'TPR@1%FPR']
print(show.to_string(index=False))

summary_path = out / 'results.json'
if summary_path.is_file():
  with open(summary_path) as f:
    s = json.load(f)
  print(f"\nSanity d_wm_to_w (identity): {s.get('sanity_d_wm_to_w', 'n/a')}")

if SAVE_TO_DRIVE:
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'Copied to Drive: {DRIVE_OUTPUT}')